# Group Project Datasets Fall 2026 (Honors)

|Helpful Links|
|---|
|[Elements of Data Science Primer](https://laserchemist.github.io/dprimer/intro.html)|
|[Inferential Thinking](https://inferentialthinking.com/chapters/intro.html)|
|[Data Science Toolkit examples](https://temple.2i2c.cloud/hub/user-redirect/lab/tree/datascience/Spring%202026/DataScience_Toolkit.ipynb)|

### Your group chooses one of the following data sets.
This notebook contains:
* The code to load each of the data sets
* References to the source and possible metadata
* Data cleaning issues to consider
* One or two ideas for relationships to explore, but do not feel constrained -- explore the data use your imagination to find other possibilities!

## Dataset Areas
<ul>
  <li><a href="#Bioinformatics">Bioinformatics</a>: Human Gut Microbiome Diversity, probably simulated (DS17), Gene Expression in Cancer vs. Normal Tissue (DS16), Protein Amino Acid Composition (DS20)</li>
  <li><a href="#Ecology">Ecology</a>: Ecological Footprint (DS1), Bird Bones and Ecological Groups (DS4)</li>
  <li><a href="#Medical">Public Health and Medical</a>: Fetal Health (DS9), Diabetes in US Adults: NHANES (DS10)</li>
  <li><a href="#Chemistry">Chemistry</a>: Periodic Table (DS3)</li>
  <li><a href="#Environmental-Science">Environmental Science</a>: Air Quality (DS5), Philadelphia PM2.5 and the 2023 Wildfire Smoke (DS21), Global Sustainable Energy (DS14), Earthquakes in the Eastern US and Canada (DS6), Pennypack Creek Stream Gauge, water year 2024 (DS7), Philadelphia Daily Weather, 1940–2025 (DS8), CAMELS Hydrology (DS19)</li>
  <li><a href="#General">General</a>: Student Performance (DS18), Philadelphia Graduation Rates (DS11), Philadelphia Crime (DS13), Staten Island Motor Vehicle Crashes, 2012–2024 (DS15)</li>
</ul>

In [2]:
# Import Numpy and Datascience modules.
import numpy as np
import pandas as pd
from datascience import *
import matplotlib.pyplot as plt
%matplotlib inline

Useful auxiliary data (can use .join() to merge with your data where appropriate)

**US population by zip code (source census.gov)**

These are **2010 census** counts. Zip codes are kept as text so that leading zeros (e.g., `00601`) are not lost; the population column is converted to numbers.

In [ ]:
zippop = Table().read_table('data/census_pop_byzip.csv', dtype=str)

# Every column loaded as text; convert population to numbers (zip codes stay text)
zippop = zippop.with_column('Total 2010 Population', zippop.column('Total 2010 Population').astype(int))

In [ ]:
zippop.where('zip code','19122')

**World population by country and year (source: UN World Population Prospects 2024)**

Five columns load as text because the numbers have spaces between digit groups (`2 514 762`); the code below converts them. Values marked "(thousands)" are in thousands of people, so `2 514 762` means about 2.5 billion. Doubling time is `...` when a population is not growing; it becomes `nan`. "World" is included as a country.

In [ ]:
worldpop = Table().read_table('data/WPP2024.csv', low_memory=False)

def text_to_number(text):
    """Remove the spaces between digit groups ('2 514 762' -> 2514762.0); '...' becomes nan."""
    if text == '...':
        return np.nan
    return float(text.replace(' ', ''))

text_columns = ['Total Population, as of 1 January (thousands)',
                'Natural Change, Births minus Deaths (thousands)',
                'Population Change (thousands)',
                'Population Annual Doubling Time (years)',
                'Births (thousands)']
for label in text_columns:
    worldpop = worldpop.with_column(label, worldpop.apply(text_to_number, label))

In [ ]:
worldpop.where('Country','France').show(3)

## **Bioinformatics Datasets** <a id="Bioinformatics"></a>

---
# Human Gut Microbiome Diversity (DS17): probably simulated data

**About this dataset.** No source, study, or collection method is given for `data/microbiome_data.csv`, and several features suggest it was generated by a computer rather than measured:
- Only 6 categories of bacteria (5 phyla plus "Other"); real sequencing studies report hundreds of taxa per sample.
- Every sample's abundances add to exactly 100%, stored to about 15 decimal places, far more precision than any measurement.
- Exactly 50 Disease and 50 Healthy samples, and no information about the disease, body site, or how samples were collected.

Treat it as a realistic-looking practice dataset, not as evidence about real people. In your report, say that the data are probably simulated, and do not generalize your findings to human health.

### Background:
The human body hosts trillions of microorganisms (bacteria, archaea, fungi) that vary by body site (gut, skin, oral cavity). The gut microbiome is particularly diverse with 500-1000 bacterial species. Microbial community composition affects health, digestion, immunity, and disease. Researchers classify bacteria by taxonomic levels (phylum, class, order, family, genus, species). Body site, diet, age, geography, and health status all influence microbiome composition. Common measurements include species richness (# of species), Shannon diversity index, and relative abundance of taxa.

### **<font color=green>You are not alone!**
Your body is home to approximately 38 trillion microorganisms—bacteria, archaea, fungi, and viruses. That's actually MORE microbial cells than human cells! Don't worry—most are helpful or harmless.
What is the microbiome?
The microbiome is the community of all microorganisms living in and on your body. Different body sites have different communities:

- Gut (digestive tract): Most diverse, 500-1000 bacterial species
- Skin: Adapted to dry, salty conditions
- Mouth: Different bacteria live on teeth, tongue, cheeks
- Other sites: Nose, lungs, reproductive tract

#### The Gut Microbiome—A Focus for Research:
Your gut microbiome:

Helps digest food (especially fiber we can't break down)
Makes vitamins (like vitamin K and some B vitamins)
Trains your immune system
Protects against harmful bacteria
Communicates with your brain (the "gut-brain axis")

- Key Bacterial Phyla in Your Gut:

Firmicutes: Large group including many beneficial species
Bacteroidetes: Break down complex carbohydrates
Proteobacteria: Includes E. coli; some helpful, some harmful
Actinobacteria: Includes Bifidobacterium (common in infants)

- What Shapes Your Microbiome?

Diet: Different foods feed different bacteria
Antibiotics: Kill bacteria indiscriminately, disrupting balance
Age: Infant, adult, and elderly microbiomes differ
Geography: People in different countries have different microbiomes
Health status: Disease can change the microbiome (and vice versa!)

- Measuring Microbiome Diversity:
Species Richness: Simply counting how many different species are present

More species = more diverse
Like counting how many different tree species are in a forest

- Shannon Diversity Index: Considers both richness AND evenness

Accounts for how evenly distributed species are
Higher number = more diverse
A forest with 100 tree species where one makes up 99% of trees is less diverse than one where species are evenly distributed

- The Firmicutes/Bacteroidetes Ratio (F/B Ratio):
This ratio has been linked to:

Obesity (higher ratio often seen in obese individuals)
Diet (changes with Western vs. traditional diets)
Various diseases

But it's controversial—not all studies agree on its importance!
Dysbiosis = Imbalance:
When the microbiome gets out of balance (dysbiosis), it's associated with:

Inflammatory bowel diseases (Crohn's, ulcerative colitis)
Obesity and metabolic syndrome
Allergies and autoimmune diseases
Mental health (depression, anxiety)
Infections (when harmful bacteria take over)

### Possible Research Questions to Explore:

- Health vs. disease: Do healthy people have more diverse microbiomes than people with disease? 
- Diet effects: Do vegetarians have different microbiome composition or diversity than omnivores?
- Age correlation: Does microbiome diversity decrease with age? 
- F/B ratio differences: Is the Firmicutes/Bacteroidetes ratio really different between healthy and disease groups?
- Body site comparison: How different is gut microbiome diversity compared to skin or oral cavity?
- Antibiotic impact (if data available): Do people who recently took antibiotics have lower diversity?
- Geographic patterns: Do people from different regions/countries have characteristic microbiome signatures?
- Stability: Do individuals have consistent microbiomes over time, or does it fluctuate?

### Authoritative Sources:

- NIH Human Microbiome Project Overview:

[https://www.hmpdacc.org/overview/](https://www.hmpdacc.org/overview/)<br>
[https://commonfund.nih.gov/hmp](https://commonfund.nih.gov/hmp)


- American Gut Project: Citizen science project that analyzed thousands of gut microbiomes

[http://americangut.org/](http://americangut.org/) 


- Harvard Medical School: The Microbiome

[https://www.hsph.harvard.edu/nutritionsource/microbiome/](https://www.hsph.harvard.edu/nutritionsource/microbiome/)


- Nature: The Human Microbiome (Review Article)

[https://www.nature.com/articles/nature11234](https://www.nature.com/articles/nature11234)


- NIH News in Health: Keeping Your Gut in Check

[https://newsinhealth.nih.gov/2017/05/keeping-your-gut-check](https://newsinhealth.nih.gov/2017/05/keeping-your-gut-check)

## Possible Questions

Questions that fit the columns in this file (`Health_Status`, `Diet`, `Age`, and the abundance of each of the 6 taxa):

- **Firmicutes/Bacteroidetes (F/B) ratio and health:** is the mean F/B ratio different for Disease and Healthy samples?

  Simulation: A/B test shuffling `Health_Status` labels; bootstrap confidence interval for the difference.

- **Diet:** do vegetarians and omnivores differ in F/B ratio or in Bacteroidetes abundance? Be careful: diet and health status are related in this sample (32 of 55 vegetarians are Healthy, compared with 18 of 45 omnivores). Check whether a diet difference still appears *within* the Healthy group and within the Disease group.

- **Shannon diversity and health:** is mean Shannon diversity different for Disease and Healthy samples? With only 6 taxa, Shannon values fall in a narrow range (about 1.45 to 1.72). If the difference is statistically significant, is it large enough to matter?

- **Age:** is age correlated with Shannon diversity or with the F/B ratio?

  Simulation: permutation test or bootstrap confidence interval for the correlation coefficient.

- **Classification:** can kNN predict `Health_Status` from the six abundances? Half the samples are Healthy, so guessing gets 50% right; compare your accuracy with that.


### Data Cleaning Issues:

- **One row per sample per taxon.** Each sample appears on 6 rows. For per-sample values such as the F/B ratio, make a table with one row per sample and one column per taxon:
  `wide = microbiome.pivot('Bacteria', 'Sample', values='Relative_Abundance', collect=sum)`
- **Relative abundances sum to 100%.** The taxa are not independent: if one goes up, the others must go down together.
- Taxa are recorded only at the phylum level, plus "Other", so species richness cannot be measured; the `species_richness` function below can return at most 6.
- No missing values.


In [3]:
# Load microbiome data
microbiome = Table.read_table('data/microbiome_data.csv')
microbiome.show(5)

Sample,Health_Status,Diet,Age,Bacteria,Relative_Abundance
Sample_0,Healthy,Omnivore,48,Firmicutes,35.7063
Sample_0,Healthy,Omnivore,48,Bacteroidetes,24.2759
Sample_0,Healthy,Omnivore,48,Proteobacteria,12.5538
Sample_0,Healthy,Omnivore,48,Actinobacteria,7.98503
Sample_0,Healthy,Omnivore,48,Verrucomicrobia,3.62324


## Helpful background code

In [ ]:
print("Total measurements:", microbiome.num_rows)
print("Unique samples:", len(np.unique(microbiome.column('Sample'))))
print("Bacterial taxa:", np.unique(microbiome.column('Bacteria')))

# Each sample appears once per taxon (6 rows), so make a table with one row per sample
# before counting samples; grouping the full table would count each sample 6 times
samples = microbiome.select('Sample', 'Health_Status', 'Diet', 'Age')
samples = samples.group(['Sample', 'Health_Status', 'Diet', 'Age']).drop('count')

# Check sample groups (.show() displays a table even when it is not the last line)
samples.group('Health_Status').show()
samples.group('Diet').show()

In [ ]:
# Visualize one sample's composition
sample1 = microbiome.where('Sample', 'Sample_0')
sample1.barh('Bacteria', 'Relative_Abundance')
plt.xlabel('Relative Abundance (%)')
plt.title('Microbial Composition - Sample_0');

### Diversity metrics for each sample
The two functions below each take one sample's relative abundances (in %). Grouping by `Sample` with a function as the second argument applies the function to each sample's abundances, giving one value per sample. Note that this file records only 6 taxa (phyla), so "richness" can be at most 6.

In [ ]:
def shannon_diversity(abundances):
    """
    Calculate Shannon diversity index
    H = -sum(p_i * log(p_i))
    where p_i is proportion of taxon i
    """
    # Convert percentages to proportions
    proportions = abundances / 100.0
    # Remove zeros to avoid log(0)
    proportions = proportions[proportions > 0]
    return -np.sum(proportions * np.log(proportions))

def species_richness(abundances, threshold=1.0):
    """Count the number of taxa above a threshold abundance (in %)"""
    return np.sum(abundances > threshold)

abundance = microbiome.select('Sample', 'Relative_Abundance')
shannon = abundance.group('Sample', shannon_diversity).relabeled('Relative_Abundance shannon_diversity', 'Shannon')
richness = abundance.group('Sample', species_richness).relabeled('Relative_Abundance species_richness', 'Richness')

# One row per sample: metadata plus both diversity metrics
samples = samples.join('Sample', shannon).join('Sample', richness)
samples.show(5)

---
# Gene Expression in Cancer vs. Normal Tissue (DS16)


## Background:
#### What is gene expression?
Your cells all have the same DNA (your genome), but they don't all look or act the same. That's because different cells "express" (turn on) different genes:

- Liver cells express liver genes
- Muscle cells express muscle genes
- Brain cells express brain genes

Gene expression = DNA → RNA → Protein
When a gene is "expressed," the cell:

Copies the DNA into RNA (transcription)
Uses RNA to make proteins (translation)
The more RNA, the more protein usually gets made

#### Measuring Gene Expression:
Scientists can measure how much RNA is made from each gene using techniques like:

- Microarrays: Check thousands of genes at once (older technology)
- RNA-sequencing (RNA-seq): Read all RNA molecules in a sample (newer, more accurate)

The results are numbers showing expression levels—higher numbers mean more of that gene's RNA was present.
What Goes Wrong in Cancer?
Cancer happens when cells grow out of control. This happens because gene expression goes haywire:
Oncogenes (cancer-causing genes):

##### Normally help cells grow and divide
- In cancer: OVEREXPRESSED (too much RNA/protein)
-   Examples: MYC, RAS, HER2
Like having your car's gas pedal stuck down

#### Tumor Suppressor Genes:

- Normally stop cells from dividing too much or repair DNA damage
- In cancer: UNDEREXPRESSED (too little RNA/protein)
- Examples: TP53 (the "guardian of the genome"), BRCA1, BRCA2
- Like having your car's brakes cut

#### Why Study Gene Expression in Cancer?

- Diagnosis: Different cancers have different expression "signatures"
- Prognosis: Expression patterns predict how aggressive a cancer is
- Treatment: Some therapies target overexpressed proteins (like Herceptin for HER2+ breast cancer)
- Understanding biology: Reveals which cellular pathways are broken

#### Fold Change:
Scientists often talk about "fold change"—how many times more (or less) a gene is expressed:

- 2-fold increase = twice as much RNA
- 0.5-fold (or 2-fold decrease) = half as much RNA
- Typically, changes >2-fold or <0.5-fold are considered biologically significant

## Authoritative Sources:

National Cancer Institute: What is Cancer?

https://www.cancer.gov/about-cancer/understanding/what-is-cancer


National Cancer Institute: Gene Expression

https://www.cancer.gov/publications/dictionaries/cancer-terms/def/gene-expression


The Cancer Genome Atlas (TCGA): Major cancer genomics project

https://www.cancer.gov/ccg/research/genome-sequencing/tcga


Nature Education: Gene Expression

https://www.nature.com/scitable/topicpage/gene-expression-14121669/


American Cancer Society: Understanding Your Cancer

https://www.cancer.org/cancer/managing-cancer/understanding-cancer.html

## Our data: GSE15852: Breast Cancer Gene Expression Dataset
**Source:** NCBI Gene Expression Omnibus, series [GSE15852](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE15852): "Expression data from human breast tumors and their paired normal tissues," Institute for Medical Research, Malaysia (published 2009). Measured with Affymetrix Human Genome U133A microarrays, which is why genes are identified by probe IDs such as `201650_at`.

### Study Design
* **43 patients, 86 samples.** Each patient contributed a sample of breast tumor **and** a sample of adjacent normal breast tissue. This is a *paired* design: every tumor has a normal sample from the same person to compare with.
* The patients were treated in Malaysia and were aged 22 to 79. Most tumors (35 of 43) were infiltrating ductal carcinoma.

### Two files
| File | One row per | Columns |
|---|---|---|
| `data/gene_expression_with_symbols.csv` | probe per sample (1,000 probes × 86 samples = 86,000 rows) | `Gene` (probe ID), `Gene_Symbol`, `Sample`, `Sample_Type` (Cancer or Normal), `Expression` |
| `data/gene_expression_samples.csv` | sample (86 rows) | `Sample`, `Patient` (links each tumor to its paired normal sample), `Sample_Type`, `Grade` (tumor grade 1–3), `Age`, `Histology` |

### Data Cleaning Issues:
* **Use the pairing.** The two samples from one patient are far more alike than samples from different patients. The `Patient` column lets you compute, for each patient, tumor minus normal. Treating the 86 samples as two independent groups of 43 ignores this and wastes information.
* **Work in log₂ units.** Expression values are raw intensities and strongly right-skewed. Take log₂ first: a difference of 1 in log₂ expression is a 2-fold change, and −1 is a halving. The code below adds a `log2_expression` column.
* **These 1,000 probes are not a random sample of genes.** The microarray has about 22,000 probes; how these 1,000 were chosen is not documented, and all of them are highly expressed. So you cannot estimate "what fraction of human genes change in cancer" from this file.
* **A probe is not a gene.** The 1,000 probes cover 812 different genes; some genes have several probes (ACTB has 11). Probes for the same gene are not independent.
* **Tumor and normal tissue are made of different cells.** Normal breast tissue is largely fat; a tumor is mostly epithelial cells. Many of the largest differences are in genes typical of fat cells (lower in tumors: `LEP`, `LPL`, `ADIPOQ`) or of epithelial cells (higher in tumors: `KRT19`, `KRT18`, `CDH1`). A difference between the two tissues can reflect *which cells are in the sample* rather than a gene being switched on or off inside cancer cells.
* The file has no survival or treatment information, and only 8 tumors are grade 1, so questions about outcomes or small subgroups cannot be answered well.

### Possible Research Questions:
* **One gene, done properly.** Choose a gene (for example `KRT19`, probe `201650_at`) and ask whether its expression differs between a patient's tumor and normal tissue. Use the per-patient log₂ fold change (tumor − normal) as in the code below, with the mean of those 43 differences as your test statistic. Under the null hypothesis the labels "Cancer" and "Normal" could be swapped *within each patient*, which is the same as randomly flipping the sign of each patient's difference. Give a bootstrap confidence interval for the mean fold change by resampling patients.
* **What does pairing buy you?** Run the test above, then repeat it as an ordinary A/B test that shuffles labels across all 86 samples. Compare the two results for a gene with a modest difference and explain why they differ.
* **Many tests at once.** Test all 1,000 probes. If no gene differed at all, about 50 would still come out "significant" at the 5% level. How many do you find? What cutoff would you use so that most of your discoveries are real?
* **Up or down, and which genes?** Among probes that change at least 2-fold, are more higher in tumors or lower? Look up the ten largest changes in each direction (https://www.genecards.org). What kinds of genes are they, and what does that say about the tissue-composition issue above?
* **Tumor grade.** Is a gene's fold change different in grade 3 tumors (12 patients) than in grade 1–2 tumors (31 patients)? The groups are small; discuss what difference you could realistically detect.
* **Classification.** Can kNN tell tumor from normal tissue using the expression of two or three genes? Half the samples are tumors, so guessing is right 50% of the time. Two cautions: keep both samples from a patient on the same side of your training/test split, and choose your genes using the training patients only. Choosing genes with all the data and then testing on some of it makes the classifier look better than it is.

In [ ]:
# Load the expression data and the sample information
expr = Table.read_table('data/gene_expression_with_symbols.csv')
samples = Table.read_table('data/gene_expression_samples.csv')

print("Total measurements:", expr.num_rows)
print("Probes:", len(np.unique(expr.column('Gene'))), "  Samples:", len(np.unique(expr.column('Sample'))))
samples.group('Grade').show()

# Work in log2 units: a difference of 1 is a 2-fold change
expr = expr.with_column('log2_expression', np.log2(expr.column('Expression')))

# Example: one probe (KRT19), with each sample's patient attached
one_gene = expr.where('Gene', '201650_at').join('Sample', samples.select('Sample', 'Patient'))

# One row per patient: log2 expression in tumor and in normal tissue, and their difference
paired = one_gene.pivot('Sample_Type', 'Patient', values='log2_expression', collect=sum)
paired = paired.with_column('log2_fold_change', paired.column('Cancer') - paired.column('Normal'))
paired.show(5)
print("Mean log2 fold change (tumor - normal):", round(np.mean(paired.column('log2_fold_change')), 2))

---
# Protein Amino Acid Composition Analysis (DS20)
The amino acid composition of proteins differs for different functional classes. Proteins are composed of 20 different amino acids, each with unique chemical properties. The proportion and distribution of these amino acids determines a protein's 3D structure and biological function. You can use simulation-based hypothesis testing to investigate whether proteins from different functional classes have significantly different amino acid compositions.


## Background - Proteins and Amino Acids

### What are proteins?

Proteins are large molecules made from chains of **amino acids**. Think of amino acids like letters in an alphabet - just as 26 letters create all English words, 20 amino acids create all proteins in living organisms.

### The 20 Standard Amino Acids

Each amino acid has:
- A **name** (e.g., Alanine, Glycine, Tryptophan)
- A **three-letter code** (Ala, Gly, Trp)
- A **one-letter code** (A, G, W)
- **Unique chemical properties**

### Chemical Properties

The key property we'll study is whether an amino acid is:
- **Hydrophobic** ("water-fearing") - avoids water, clusters in protein interior
- **Hydrophilic** ("water-loving") - comfortable in water, found on protein surfaces
- **Charged** - carries positive or negative charge
- **Special** - unique structural roles (flexibility, bonding, etc.)

### Why Composition Matters

- **Membrane proteins** need more hydrophobic amino acids (to sit in fatty membrane)
- **Enzymes** need charged amino acids at active sites (for catalysis)
- **Structural proteins** need special amino acids (for flexibility or strength)
- **Transport proteins** need specific arrangements (to bind and release cargo)

### Example Protein Sequence

```
MKTAYIAKQRQISFVKSHFSRQLEERLGLIEVQAPILSRVGDGTQDN...
```

Each letter represents one amino acid. This sequence contains all information needed to determine the protein's 3D structure!

## Visualizing Amino Acids (Connection to Lab 10)

### Load Amino Acid Reference Data

Let's load a table with information about all 20 amino acids, including their **SMILES** strings (chemical structure notation) that we can visualize in 3D.

In [4]:
# Load amino acid reference table
amino_acids = Table.read_table('data/amino_acids.csv')
amino_acids

Name,Three_Letter,One_Letter,SMILES,Property,Description
Alanine,Ala,A,C[C@H](N)C(O)=O,Hydrophobic,"Small, nonpolar"
Arginine,Arg,R,NC(NCCCC(N)C(O)=O)=[NH2+],Charged (+),"Basic, positively charged"
Asparagine,Asn,N,NC(=O)CC(N)C(O)=O,Hydrophilic,"Polar, uncharged"
Aspartic acid,Asp,D,OC(=O)CC(N)C(O)=O,Charged (-),"Acidic, negatively charged"
Cysteine,Cys,C,SC[C@H](N)C(O)=O,Special,Forms disulfide bonds
Glutamine,Gln,Q,NC(=O)CCC(N)C(O)=O,Hydrophilic,"Polar, uncharged"
Glutamic acid,Glu,E,OC(=O)CCC(N)C(O)=O,Charged (-),"Acidic, negatively charged"
Glycine,Gly,G,NCC(O)=O,Special,"Smallest, most flexible"
Histidine,His,H,NC(Cc1cnc[nH]1)C(O)=O,Charged (+),Can be charged or neutral
Isoleucine,Ile,I,CC[C@H](C)[C@H](N)C(O)=O,Hydrophobic,"Branched, nonpolar"


### Visualize Amino Acid Structures

Using the SMILES strings and RDKit (same library as Lab 10), we can display the 2D and 3D structures of amino acids.

In [5]:
# Import RDKit for molecular visualization (like Lab 10)
try:
    from rdkit import Chem
    from rdkit.Chem import Draw
    from rdkit.Chem import AllChem
    rdkit_available = True
except ImportError:
    print("RDKit not available - skipping 3D visualization")
    rdkit_available = False

RDKit not available - skipping 3D visualization


In [6]:
def display_amino_acid_2D(one_letter_code):
    """Display 2D structure of an amino acid given its one-letter code"""
    if not rdkit_available:
        print("RDKit not available")
        return
    
    # Get SMILES for this amino acid
    aa_row = amino_acids.where('One_Letter', one_letter_code)
    if aa_row.num_rows == 0:
        print(f"Amino acid {one_letter_code} not found")
        return
    
    name = aa_row.column('Name').item(0)
    smiles = aa_row.column('SMILES').item(0)
    property_type = aa_row.column('Property').item(0)
    
    # Create molecule and display
    mol = Chem.MolFromSmiles(smiles)
    if mol:
        print(f"{name} ({one_letter_code}) - {property_type}")
        return Draw.MolToImage(mol, size=(300, 300))
    else:
        print(f"Could not create molecule for {name}")

# Example: Display Alanine (hydrophobic) and Lysine (charged)
if rdkit_available:
    print("Hydrophobic example:")
    display(display_amino_acid_2D('A'))  # Alanine
    
    print("\nCharged example:")
    display(display_amino_acid_2D('K'))  # Lysine

**<font color='#9E1B34'>Explore:</font>** Try displaying other amino acids:
- Hydrophobic: V, I, L, M, F, W, P
- Hydrophilic: S, T, N, Q
- Charged positive: K, R, H
- Charged negative: D, E
- Special: G (smallest), C (forms bonds), Y (aromatic)

## Load and Explore the Protein Dataset

### The Dataset

We have 200 real human proteins from **UniProt** (the reviewed Swiss-Prot section), 50 drawn at random from each of four functional classes:
- **Enzymes** (50) - catalyze chemical reactions; soluble (not in a membrane)
- **Membrane proteins** (50) - embedded in cell membranes but not transporters; mostly receptors and adhesion proteins
- **Structural proteins** (50) - provide mechanical support: cytoskeleton, extracellular matrix, skin
- **Transport proteins** (50) - membrane transporters and channels that move molecules and ions across membranes

Note that Membrane and Transport proteins *both* sit in membranes. Exactly how each class was defined, and why it matters, is in `data/protein_sequences_uniprot_README.md`. Read it before you choose a question.

Each protein includes:
- `Protein_ID`: a real UniProt accession. Look any protein up at `https://www.uniprot.org/uniprotkb/<Protein_ID>`
- `Name` and `Gene`
- `Function_Class`
- `Sequence` (one-letter codes) and `Length`
- `Organism` (Human for every protein)

In [8]:
# Load the protein dataset
proteins = Table.read_table('data/protein_sequences_uniprot.csv')
proteins.show(5)

Protein_ID,Name,Function_Class,Sequence,Length,Organism
P00001,Lactase,Enzyme,DKMLLGHDVQGAVLTHCAHLWTQYAKHWRMIQQGSWDNFCMQPDQKHMQGQGHQMV ...,399,Yeast
P00002,Amylase,Enzyme,QKYCYSMQWNQQHDTWQDYYGMNLRCYQPPGHPIDMYDLMQTTGQQYCVYRCQWQW ...,336,C. elegans
P00003,Pepsin,Enzyme,STTEHVGTKQDAQECPYDKYYPSLELRGGTYIGATYMPDEEERPQVQRNCQAKKHA ...,324,Human
P00004,Trypsin,Enzyme,TCDWPNWIWVPQNSELQYLNGQCHFQWQDKRVLAAYTIQGQQLNGDMGIMKKHAGA ...,480,Mouse
P00005,Catalase,Enzyme,RMDQKYGVFKSYQTPYIGSEAETKLDYMWPEKHLNRSQMDNQEEQFIEHPFMGWSW ...,489,C. elegans


In [9]:
# Examine the data structure
print("Column labels:", proteins.labels)
print("Number of proteins:", proteins.num_rows)

Column labels: ('Protein_ID', 'Name', 'Function_Class', 'Sequence', 'Length', 'Organism')
Number of proteins: 200


In [10]:
# How many proteins in each functional class?
proteins.group('Function_Class')

Function_Class,count
Enzyme,50
Membrane,50
Structural,50
Transport,50


### Examining a Protein Sequence

Let's look at one protein's sequence in detail:

In [11]:
# Get first protein
example = proteins.row(0)
print("Protein:", example.item('Name'))
print("Class:", example.item('Function_Class'))
print("Length:", example.item('Length'), "amino acids")
print("\nSequence:")
print(example.item('Sequence'))

Protein: Lactase
Class: Enzyme
Length: 399 amino acids

Sequence:
DKMLLGHDVQGAVLTHCAHLWTQYAKHWRMIQQGSWDNFCMQPDQKHMQGQGHQMVLSQMYNIRFGMQGLIYEEGFSLGEQMSTEDYQPDYKGCHNWWFDTHQVSGIIWVQYEYGVYHYGNWIQGYIHSMLTQTHDCQQASNMLKKVVQVITQIEYLDNEERPGTLVRDLHLDVDTQANVQYPDGHHPLYQDSSEYYYEAVGRFVDLLHEDWDMQDDYDKASLIQQYDYKKQAEMPEWQQQNLQCQQGACYCIKQDKVHQWKGDDECFYGYQRRGEEYGSQVKLCRIRVLYNIRYAEVIRKDDQRCGFTMPQGDQMAARDEIQEHKSVTGFIWIGENANTLSNHEITDKYDIFELRQHSESRQTQQMLLALEFNGMYWFDDYDYAVYYQQCGQQYSYGL



## Key Function - Calculating Amino Acid Composition

### The Core Function

This function calculates what percentage of a sequence belongs to a specific group of amino acids.

In [12]:
def calculate_aa_composition(sequence, aa_group):
    """
    Calculate the percentage of amino acids in a sequence that belong to a specific group.
    
    Parameters:
    sequence (str): The amino acid sequence (e.g., 'MKTAYIAK...')
    aa_group (str): String of amino acid letters to count (e.g., 'AVILMFWP' for hydrophobic)
    
    Returns:
    float: Percentage (0-100) of sequence made up of amino acids in the group
    
    Example:
    >>> calculate_aa_composition('MVLSPADKTNC', 'AVILMFWP')
    45.45  # 5 out of 11 amino acids are hydrophobic
    """
    count = 0
    for aa in aa_group:
        count = count + sequence.count(aa)   # how many times this letter appears
    return 100 * count / len(sequence)

# Test the function
test_seq = "MVLSPADKTNC"
hydrophobic_aas = "AVILMFWP"
print("Test sequence:", test_seq)
print("Hydrophobic percentage:", round(calculate_aa_composition(test_seq, hydrophobic_aas), 2))

Test sequence: MVLSPADKTNC
Hydrophobic percentage: 45.5%


### Apply to All Proteins

Let's calculate hydrophobic percentage for ALL proteins in our dataset:

In [17]:
# Define amino acid groups
hydrophobic = "AVILMFWP"
hydrophilic = "STNQC"
charged_positive = "KRH"
charged_negative = "DE"

# .apply() needs a function of one argument (the sequence), so wrap the general function
def hydrophobic_percent(sequence):
    return calculate_aa_composition(sequence, hydrophobic)

# Calculate hydrophobic percentage for each protein using apply()
proteins = proteins.with_column('Hydrophobic_%', proteins.apply(hydrophobic_percent, 'Sequence'))

# Display result
proteins

Protein_ID,Name,Function_Class,Sequence,Length,Organism,Hydrophobic_%
P00001,Lactase,Enzyme,DKMLLGHDVQGAVLTHCAHLWTQYAKHWRMIQQGSWDNFCMQPDQKHMQGQGHQMV ...,399,Yeast,31.5789
P00002,Amylase,Enzyme,QKYCYSMQWNQQHDTWQDYYGMNLRCYQPPGHPIDMYDLMQTTGQQYCVYRCQWQW ...,336,C. elegans,31.8452
P00003,Pepsin,Enzyme,STTEHVGTKQDAQECPYDKYYPSLELRGGTYIGATYMPDEEERPQVQRNCQAKKHA ...,324,Human,31.1728
P00004,Trypsin,Enzyme,TCDWPNWIWVPQNSELQYLNGQCHFQWQDKRVLAAYTIQGQQLNGDMGIMKKHAGA ...,480,Mouse,30.4167
P00005,Catalase,Enzyme,RMDQKYGVFKSYQTPYIGSEAETKLDYMWPEKHLNRSQMDNQEEQFIEHPFMGWSW ...,489,C. elegans,30.0613
P00006,DNA Polymerase,Enzyme,MVMFGKKPGEFEHPCTLQYESCDKLHGTFYVLPSYKPGYYACQYAWNQHTVDYNGQ ...,342,E. coli,29.8246
P00007,RNA Polymerase,Enzyme,GGQTQHTHEENVSAYQFMDDSIQQEKSVQCDFQARQQVLCIWGTMQMQTGKYQWQQ ...,489,Mouse,30.6748
P00008,Kinase,Enzyme,CGQYTGTSWKPCGMIVQHTGGQEYDFGHMSAFYMCKMQPQCFAEQFDEGCSTDGAT ...,324,C. elegans,29.9383
P00009,Phosphatase,Enzyme,HQYMNKKAYDGYDCKPDFDLFFIDERAMLLGFNVAKRPWTWPQRHQSRDDPKSYDR ...,235,Human,31.4894
P00010,Lipase,Enzyme,RQYNECWHQQTRPQQEYEKMSEDQQIHDDDLMEHADQDGALYHEYYQRQNLRMQHK ...,213,Mouse,31.9249



## A Possible Starting Question: Membrane vs Enzyme Proteins

**Do membrane proteins have higher hydrophobic content than enzymes?**

**Biological Rationale:**  
Membrane proteins must interact with the fatty (hydrophobic) lipid bilayer of cell membranes. Enzymes typically work in the watery cytoplasm. Therefore, we predict membrane proteins will have more hydrophobic amino acids.

The `Hydrophobic_%` column above gives you what you need to start. Your group decides the rest: how to visualize the two groups, the null and alternative hypotheses, the test statistic, and how you will judge whether the difference is large enough to matter biologically, not only whether it is statistically significant.

### Suggested Research Questions


#### Question A: Charged Amino Acids in Enzymes
- **Question:** Do enzymes have more charged amino acids (K, R, H, D, E) than structural proteins?
- **Rationale:** Enzymes need charged residues at active sites for catalysis
- **Groups to compare:** Enzyme vs Structural
- **Feature to calculate:** Total charged percentage (positive + negative)

#### Question B: Small/Flexible Amino Acids in Structural Proteins
- **Question:** Do structural proteins have more glycine and proline (G, P) than other proteins?
- **Rationale:** Glycine (smallest) allows tight packing; proline creates kinks  
- **Groups to compare:** Structural vs Transport
- **Feature to calculate:** G + P percentage

#### Question C: Hydrophilic Amino Acids in Transport Proteins
- **Question:** Do transport proteins have more hydrophilic amino acids (S, T, N, Q, C) than membrane proteins?
- **Rationale:** Transport proteins often work in aqueous environments
- **Groups to compare:** Transport vs Membrane
- **Feature to calculate:** Hydrophilic percentage

#### Question D: Protein Length by Function
- **Question:** Are some protein classes significantly longer than others?
- **Rationale:** Functional complexity may correlate with protein length
- **Groups to compare:** Any two functional classes
- **Feature to analyze:** Length column (already in table)




## References and Further Reading

- **UniProt Database:** https://www.uniprot.org - Comprehensive protein sequence and functional information. Data in this project: UniProtKB/Swiss-Prot release 2026_04.
- The UniProt Consortium (2025). "UniProt: the Universal Protein Knowledgebase in 2025." *Nucleic Acids Research* 53:D609-D617.
- **Protein Data Bank (PDB):** https://www.rcsb.org - 3D protein structures
- Kyte, J. & Doolittle, R.F. (1982). "A simple method for displaying the hydropathic character of a protein." *J. Mol. Biol.* 157:105-132.
- **NCBI Protein Database:** https://www.ncbi.nlm.nih.gov/protein

---
---
## **Ecological Datasets** <a id="Ecology"></a>

## Ecological Footprint (DS1)
This dataset is the **National Footprint Accounts, 2019 edition**, from the Global Footprint Network: how much biologically productive land and water each country uses (its ecological footprint) and how much it has (its biocapacity), for 1961 to 2016. More information: https://data.world/footprint/nfa-2019-edition and https://www.footprintnetwork.org

### How the file is organized: the `record` column
Each country-year appears **8 times**, one row per `record` type. The land-type columns (`crop_land`, `grazing_land`, `forest_land`, `fishing_ground`, `built_up_land`, `carbon`) and `total` mean something different, in different units, in each:

| `record` | What the numbers measure | Units |
|---|---|---|
| `AreaTotHA` | physical area of each land type | hectares (ha) |
| `AreaPerCap` | physical area per person | ha per person |
| `BiocapTotGHA` | biocapacity: what the land can produce | global hectares (gha) |
| `BiocapPerCap` | biocapacity per person | gha per person |
| `EFProdTotGHA` | footprint of what the country **produces** | gha |
| `EFProdPerCap` | production footprint per person | gha per person |
| `EFConsTotGHA` | footprint of what the country **consumes** (production + imports − exports) | gha |
| `EFConsPerCap` | consumption footprint per person | gha per person |

A **global hectare** is a hectare of world-average biological productivity. Converting to global hectares is what makes it possible to add cropland, forest, and fishing grounds together in `total`. `carbon` is the land that would be needed to absorb CO₂ emissions; it is 0 in the area and biocapacity records.

**Always select one `record` type before you compare, average, or plot.** Mixing records adds hectares to global hectares and totals to per-person values.

### Data Cleaning Issues:
* **One malformed row** (Moldova, 2014, `EFConsPerCap`) is cut off: its quality score `3A` landed in `forest_land`, so the whole `forest_land` column loads as text. The code below drops that row and converts the column to numbers.
* **Missing land-type breakdowns:** for 58 countries (including New Zealand and Egypt), the land-type columns are blank in every year and only `total` is given. Analyses by land type will silently leave these countries out.
* **Countries change over time:** the USSR, Yugoslavia, and Czechoslovakia end around 1991–1992 and their successor states begin; Sudan splits into Sudan and South Sudan in 2011. A country's time series may not cover 1961–2016.
* **"World"** is included as a country. Leave it out of comparisons between countries.
* `QScore` is a data-quality score for each country-year; the scale is explained here: https://www.footprintnetwork.org/data-quality-scores/

### Possible Hypotheses to Test:
* **Ecological deficit:** a country whose consumption footprint (`EFConsPerCap` total) is larger than its biocapacity (`BiocapPerCap` total) is running a deficit. Did the share of countries in deficit change between 1961 and 2016?
* **Consumption vs. production:** which countries consume more than they produce (net importers of footprint)? Is that related to population or to biocapacity?
* **Land-use change over time:** China is a fascinating example where huge policy shifts drove change: https://www.ncbi.nlm.nih.gov/pmc/articles/PMC5036680/


In [ ]:
url = 'data/NFA 2019 public_data.csv'
ecoFootprint = Table.read_table(url, low_memory=False)

# Drop the one malformed row (its 'forest_land' holds the text '3A'), then make forest_land numeric
ecoFootprint = ecoFootprint.where('forest_land', are.not_equal_to('3A'))
ecoFootprint = ecoFootprint.with_column('forest_land', ecoFootprint.column('forest_land').astype(float))

# Each country-year has 8 rows, one per record type
ecoFootprint.group('record').show()

# Example: select one record type before analyzing
footprint_per_person = ecoFootprint.where('record', 'EFConsPerCap')
footprint_per_person.where('country', 'China').show(3)

---
## Bird Bones and Ecological Groups (DS4)
Can you tell how a bird lives from its skeleton? This dataset has measurements of five bones from **420 bird skeletons**, each labeled with the bird's ecological group (swimmer, wader, raptor, and so on). Wing bones (humerus, ulna) relate to flight; leg bones (femur, tibiotarsus, tarsometatarsus) relate to walking, perching, wading, and grabbing prey.

**Source:** posted on Kaggle as "Birds' Bones and Living Habits" (https://www.kaggle.com/datasets/zhangjuefei/birds-bones-and-living-habits). Secondary descriptions credit skeletons in the Natural History Museum of Los Angeles County, covering about 245 species; the file itself does not name species or document how the birds were chosen.

**Columns** (all measurements in millimeters):

| Column | Bone | Measurement |
|---|---|---|
| `huml`, `humw` | humerus (upper wing) | length, width (diameter) |
| `ulnal`, `ulnaw` | ulna (forearm of the wing) | length, width |
| `feml`, `femw` | femur (thigh) | length, width |
| `tibl`, `tibw` | tibiotarsus (lower leg) | length, width |
| `tarl`, `tarw` | tarsometatarsus (the "foot bone" above the toes) | length, width |
| `type` | ecological group | see below |

| `type` | Group | Birds |
|---|---|---|
| `SO` | singing birds (songbirds) | 128 |
| `SW` | swimming birds | 116 |
| `W` | wading birds | 65 |
| `R` | raptors (birds of prey) | 50 |
| `P` | scansorial birds (climbers, such as woodpeckers) | 38 |
| `T` | terrestrial (ground-dwelling) birds | 23 |

### Data Cleaning Issues:
* **7 birds have one or more missing measurements** (blank, loaded as `nan`). The code below shows how to keep only complete rows.
* **One value is almost certainly a typo:** bird `id` 0 has a tibiotarsus length of 5.5 mm but a femur of 41.8 mm; in every other bird the tibiotarsus is about as long as the femur or longer. Decide how to handle it and say so in your report.
* **Body size dominates.** Big birds have big bones: humerus length ranges from about 10 mm to 420 mm (an albatross-sized swimmer). Comparing raw lengths mostly compares body size. Ratios such as humerus length ÷ femur length (wing vs. leg) describe *shape* and are often more informative. The very large swimmers also make the distributions skewed.
* **Groups are unequal in size** (128 songbirds, 23 terrestrial birds). A classifier that always guesses "songbird" is right 30% of the time.
* **Birds are not independent.** There are about 245 species among 420 birds, so many species appear more than once. Two skeletons of the same species are very similar, which can make a classifier look better than it really is.

### Possible Hypotheses:
* **Classification:** can kNN predict a bird's ecological group from its bones? Compare your accuracy with the 30% baseline, and look at which groups get confused with each other.
* **Shape, not size:** do swimming birds have longer wings relative to their legs (`huml` ÷ `feml`) than wading birds? (A/B test)
* **Raptors' feet:** are raptors' tarsometatarsi thicker relative to their length (`tarw` ÷ `tarl`) than songbirds'?
* **Scaling:** how well does humerus length predict ulna length? Is the relationship the same for every group? (regression)

In [ ]:
birds = Table.read_table('data/bird.csv')
birds.show(3)

# How many birds are in each ecological group?
birds.group('type').show()

# Keep only birds with all 10 measurements. A comparison with nan is always False,
# so are.above(0) removes rows where that measurement is missing.
complete = birds
for label in birds.labels[1:11]:
    complete = complete.where(label, are.above(0))
print('Birds with all measurements:', complete.num_rows)

# Example of a shape measure: wing length relative to leg length
complete = complete.with_column('hum_fem_ratio', complete.column('huml') / complete.column('feml'))

---
---
## **Public Health and Medical** <a id="Medical"></a>

---
# Fetal Health (DS9)
Kaggle Dataset: https://www.kaggle.com/datasets/andrewmvd/fetal-health-classification

Reduction of child mortality is reflected in several of the United Nations' Sustainable Development Goals and is a key indicator of human progress.
The UN expects that by 2030, countries end preventable deaths of newborns and children under 5 years of age, with all countries aiming to reduce under‑5 mortality to at least as low as 25 per 1,000 live births.

Parallel to notion of child mortality is of course maternal mortality, which accounts for 295 000 deaths during and following pregnancy and childbirth (as of 2017). The vast majority of these deaths (94%) occurred in low-resource settings, and most could have been prevented.

In light of what was mentioned above, Cardiotocograms (CTGs) are a simple and cost accessible option to assess fetal health, allowing healthcare professionals to take action in order to prevent child and maternal mortality. The equipment itself works by sending ultrasound pulses and reading its response, thus shedding light on fetal heart rate (FHR), fetal movements, uterine contractions and more.

Data
This dataset contains 2126 records of features extracted from Cardiotocogram exams, which were then classified by three expert obstetricians into 3 classes:

fetal_health
* Normal (1)
* Suspect (2)
* Pathological (3)



### Data Cleaning Issues:

The data appear to be clean. Need to research the features obtained from Cardiotocograms.

### Possible Hypotheses:
Apart from explored correlations, this dataset would be an excellent one to try k-nearest neighbors (kNN) classification to predict fetal health.

**Baseline to beat:** 1,655 of the 2,126 exams (78%) are Normal, so a "classifier" that always predicts Normal is already 78% accurate while never catching a Suspect or Pathological case. Compare your kNN accuracy with 78%, and check how well it identifies the Suspect and Pathological cases, not only its overall accuracy.

In [ ]:
filename = "data/fetal_health.csv"
fetal = Table().read_table(filename)
fetal.show(3)

---
# Diabetes in US Adults: NHANES (DS10)
Data from the **National Health and Nutrition Examination Survey (NHANES)**, run by the CDC's National Center for Health Statistics. NHANES is a nationally representative survey: participants are selected by a probability sample, interviewed at home, and then examined in a mobile examination center where blood is drawn and body measurements are taken. This file combines seven public NHANES files from the **August 2021 – August 2023** cycle.

- NHANES home: https://wwwn.cdc.gov/nchs/nhanes/
- Variable descriptions: `data/nhanes_diabetes_data_dictionary.csv` (loaded below)
- How the file was built, who is included, and known issues: `data/nhanes_diabetes_README.md`
- Published national estimates from the same data: [NCHS Data Brief 516](https://www.cdc.gov/nchs/products/databriefs/db516.htm)

### Background
Diabetes means blood sugar (glucose) stays too high. Two lab tests are used to diagnose it:
- **HbA1c** (glycohemoglobin, %): average blood sugar over the past ~3 months. Normal < 5.7, prediabetes 5.7–6.4, diabetes ≥ 6.5.
- **Fasting plasma glucose** (mg/dL), measured after not eating for 8+ hours. Normal < 100, prediabetes 100–125, diabetes ≥ 126.

About 1 in 6 US adults has diabetes, and roughly 1 in 4 of them have not been diagnosed. NHANES can find those undiagnosed cases because it measures everyone's blood, not only people who see a doctor.

Each row is one adult (age 20+, not pregnant). Lab values and body measurements were **measured**; diagnosis, medication, smoking, education, and income were **self-reported**.

### Data Cleaning Issues:
* **Fasting glucose is blank for about half the rows.** Only a random subsample was assigned to a morning fasting exam. This is by design, not because of anything about the person.
* **Survey design:** NHANES deliberately oversamples some groups (e.g., older adults), so percentages computed from the rows are not US percentages. The `sample_weight` column gives the number of US adults each person represents. You may ignore the weights, but say so in your limitations.
* **`glycemic_status` is derived**, not measured: it combines the self-reported diagnosis with the HbA1c and glucose cutoffs above.
* **Treatment changes the lab values.** People taking insulin or diabetes pills (`diabetes_medication`) can have near-normal HbA1c.
* Age is top-coded at 80 (80 means 80 or older); income-to-poverty ratio is capped at 5. Missing numbers load as `nan`; missing categories are `Unknown`.

### Possible Hypotheses:
* Is HbA1c correlated with waist circumference? With BMI? Which is the better predictor? (correlation, regression, bootstrap CI for the slope)
* Among adults whose lab values show diabetes, is the fraction **undiagnosed** different for people below vs. above the poverty line (`income_poverty_ratio` < 1)? (A/B test)
* Do current smokers have higher mean HbA1c than never smokers? Be careful: smokers and non-smokers also differ in age. How could you check whether age explains the difference?
* Can kNN predict diabetes from age, waist, BMI, and blood pressure? Do **not** use `hba1c` or `fasting_glucose` as features if your label is defined from them; that just rediscovers the cutoffs. Compare your accuracy with the accuracy of always guessing "no diabetes."


In [ ]:
url = 'data/nhanes_diabetes_2021_2023.csv'
nhanes = Table.read_table(url)
nhanes.show(3)

In [ ]:
# What each column means (pandas shows the full text; the Table display cuts long notes short)
nhanes_dict = Table.read_table('data/nhanes_diabetes_data_dictionary.csv')
pd.set_option('display.max_colwidth', None)
nhanes_dict.to_df()

In [ ]:
nhanes.group('glycemic_status')

__________________________

## **Chemistry** <a id="Chemistry"></a>

---
# Periodic Table (DS3)

<img src="data/xkcd_periodic_table.png" width="600">

The file `data/Periodic Table of Elements.csv` has one row for each of the 118 elements and 28 columns of properties. It is a widely shared compilation (posted on GitHub and Kaggle) rather than an official reference, so check any surprising value against a source such as the CRC Handbook or https://pubchem.ncbi.nlm.nih.gov/ptable/. Of course, there are numerous references that discuss element groupings.

**Units of the main numeric columns:**

| Column | Units |
|---|---|
| `AtomicMass` | atomic mass units (u) |
| `AtomicRadius` | ångströms (Å) |
| `Electronegativity` | Pauling scale |
| `FirstIonization` | electron volts (eV) |
| `Density` | g/cm³ |
| `MeltingPoint`, `BoilingPoint` | kelvin (K) |
| `SpecificHeat` | J/(g·K) |
| `Year` | year of discovery |

### Data Cleaning Issues:
* **Blank values have meanings.** `Year` is blank for 11 elements known since ancient times (e.g., gold, iron). `MeltingPoint` and `BoilingPoint` are blank for 20 elements, mostly short-lived synthetic ones that have never been measured; helium has no melting point at normal pressure. `Group` is blank for 28 lanthanides and actinides (cerium–lutetium and thorium–lawrencium).
* Three element names end with a stray space (`'Darmstadtium '`, `'Roentgenium '`, `'Copernicium '`), so `elements.where('Element', 'Copernicium')` finds nothing. Use `Symbol` to select elements, or strip the spaces.
* The yes/no columns (`Radioactive`, `Natural`, `Metal`, `Nonmetal`, `Metalloid`) contain `yes` or are blank; blank means no.
* **This is the whole population, not a sample.** There are only 118 elements and you have all of them. Think about what a p-value would mean here before running a hypothesis test; describing and modeling trends (regression, kNN) may make more sense.

### Possible Hypotheses:
* How well does atomic mass predict melting point or boiling point? Does the relationship differ for metals and nonmetals?
* How do first ionization energy and atomic radius change across a period and down a group? Fit a line within one period.
* Can kNN classify an element as a metal or nonmetal from its electronegativity, ionization energy, and density?

In [ ]:
url = 'data/Periodic Table of Elements.csv'
elements = Table.read_table(url)
elements.show(3)

---
---
## **Environmental Science** <a id="Environmental-Science"></a>

---
# Air Quality (DS5)

[From Kaggle:](https://www.kaggle.com/datasets/tawfikelmetwally/air-quality-dataset?resource=download)

**Content**
The dataset contains 9358 instances of hourly averaged responses from an array of 5 metal oxide chemical sensors embedded in an Air Quality Chemical Multisensor Device. The device was located on the field in a significantly polluted area, at road level,within an Italian city.
Data were recorded from March 2004 to February 2005 (one year)representing the longest freely available recordings of on field deployed air quality chemical sensor devices responses.

Ground Truth hourly averaged concentrations for CO, Non Metanic Hydrocarbons, Benzene, Total Nitrogen Oxides (NOx) and Nitrogen Dioxide (NO2) and were provided by a co-located reference certified analyzer. Evidences of cross-sensitivities as well as both concept and sensor drifts are present as described in De Vito et al., Sens. And Act. B, Vol. 129,2,2008 (citation required) eventually affecting sensors concentration estimation capabilities.

**Attribute Information:**

* 0-Date (DD/MM/YYYY)
* 1-Time (HH.MM.SS)
* 2-True hourly averaged concentration CO in mg/m^3 (reference analyzer)
* 3-PT08.S1 (tin oxide) hourly averaged sensor response (nominally CO targeted)
* 4-True hourly averaged overall Non Metanic HydroCarbons concentration in microg/m^3 (reference analyzer)
* 5-True hourly averaged Benzene concentration in microg/m^3 (reference analyzer)
* 6-PT08.S2 (titania) hourly averaged sensor response (nominally NMHC targeted)
* 7-True hourly averaged NOx concentration in ppb (reference analyzer)
* 8-PT08.S3 (tungsten oxide) hourly averaged sensor response (nominally NOx targeted)
* 9-True hourly averaged NO2 concentration in microg/m^3 (reference analyzer)
* 10-PT08.S4 (tungsten oxide) hourly averaged sensor response (nominally NO2 targeted)
* 11-PT08.S5 (indium oxide) hourly averaged sensor response (nominally O3 targeted)
* 12-Temperature in °C
* 13-Relative Humidity (%)
* 14-AH Absolute Humidity

### Data Cleaning Issues:
* **Missing values are coded as −200, not left blank.** Any plot or mean that includes them is badly wrong: with the −200s left in, the mean CO concentration comes out −34.2 mg/m³ instead of 2.15, and the mean temperature 9.8 °C instead of 18.3 °C. Counts in this file:

| Column | Rows with −200 (of 9,357) |
|---|---|
| `NMHC(GT)` | 8,443 (90%) |
| `CO(GT)` | 1,683 |
| `NO2(GT)` | 1,642 |
| `NOx(GT)` | 1,639 |
| every other measurement column | 366 |

  The code below replaces −200 with `nan`. Use `NMHC(GT)` only with great care: 90% of it is missing.
* **114 blank rows** at the end of the file and **two empty columns** (`Unnamed: 15`, `Unnamed: 16`). The code below removes them.
* Temperature (`T`) has 13 real negative readings (winter nights); only −200 means missing.
* The data run from 10 March 2004 to 4 April 2005, hourly. Working with time series data can be tricky using tables; look at Lab 04 for useful functions.

### Possible Hypotheses to Test:

One could test whether there is a significant difference between nitrogen oxide (`NOx(GT)`) levels in the summer vs winter months. Exploring correlations between different contaminants would also be interesting. This is a rich data set, so there are many possibilities.

In [ ]:
url = 'data/Air Quality.csv'
air = Table.read_table(url)

# Remove the two empty columns and the blank rows at the end of the file
air = air.drop('Unnamed: 15', 'Unnamed: 16')
air = air.where('Date', are.not_equal_to('nan'))

# Missing measurements are coded as -200: replace them with nan
for label in air.labels[2:]:          # every column after Date and Time
    values = air.column(label)
    air = air.with_column(label, np.where(values == -200, np.nan, values))

air.show(3)

---
# Philadelphia Air Pollution and the 2023 Wildfire Smoke (DS21)
In early June 2023, smoke from hundreds of wildfires in Quebec, Canada drifted south over the eastern United States. On 7 June the sky over Philadelphia turned a hazy orange, and fine-particle pollution rose to more than 20 times its usual level. This dataset lets you measure how large that event was compared with ordinary days, a natural experiment in your own city.

**Data:** daily **PM2.5** (fine particles 2.5 micrometers or smaller, which reach deep into the lungs) at Philadelphia's six air-monitoring sites, 2021–2024, from the US EPA Air Quality System. The monitors are run by the City of Philadelphia Air Management Services. How the file was built, site details, and known issues: `data/philly_pm25_README.md`.

| Column | Meaning |
|---|---|
| `date` | local date (YYYY-MM-DD) |
| `site_id`, `site_name` | monitoring site (loads as a number, e.g., `4` for site 0004) |
| `latitude`, `longitude` | site location |
| `pm25` | 24-hour average PM2.5, micrograms per cubic meter (µg/m³) |
| `aqi` | EPA Air Quality Index for that day |

**For reference:** EPA's 24-hour standard for PM2.5 is 35 µg/m³ and the annual standard is 9.0 µg/m³. On a typical Philadelphia day PM2.5 is around 7–10 µg/m³; on 7 June 2023 the highest site reached 239.8 µg/m³.

### Data Cleaning Issues:
* **One row per site per day** (8,192 rows). For a single citywide value per day, average the sites, as in the code below.
* Some days are missing at some sites (monitor down), and site 0004 starts in October 2021. There are no blank values.
* **Days are not independent.** Pollution on one day is closely related to the day before, and smoke arrives in episodes lasting several days. A simulation that treats days as independent draws will make your estimate look more precise than it is; say so in your limitations.
* Use `pm25`, not `aqi`, for averages and comparisons: the AQI is a nonlinear index, and EPA revised its PM2.5 breakpoints in 2024.
* On 1 July 2023 EPA changed how it delivers the instrument correction for these monitors. The values are meant to be comparable, but note it if your comparison crosses that date.

### Possible Hypotheses:
* How much higher was PM2.5 in June 2023 than in June of 2021, 2022, and 2024? Give an estimate with a confidence interval, and discuss the independence issue above.
* How many days each year exceeded the 24-hour standard of 35 µg/m³, and which events caused them? (Hint: look at July 2021 too.)
* Did all six sites experience the smoke equally? Are some sites consistently higher than others on ordinary days?
* Without the smoke episodes, would 2023's annual average have been different from the other years'? How much did a few days change the yearly number?

In [ ]:
pm = Table.read_table('data/philly_pm25_2021_2024.csv')
pm.show(3)

# One citywide value per day: the average of all sites reporting that day
city = pm.select('date', 'pm25').group('date', np.mean).relabeled('pm25 mean', 'pm25')

# The five worst days, 2021-2024
city.sort('pm25', descending=True).show(5)

# Example: just June 2023
june_2023 = city.where('date', are.containing('2023-06'))
print('Mean PM2.5 in June 2023:', round(np.mean(june_2023.column('pm25')), 1), 'µg/m³')

---
# Global Sustainable Energy Production (DS14)
Data set taken from Kaggle:

"Uncover this dataset showcasing sustainable energy indicators and other useful factors across all countries from 2000 to 2020. Dive into vital aspects such as electricity access, renewable energy, carbon emissions, energy intensity, Financial flows, and economic growth. Compare nations, track progress towards Sustainable Development Goal 7, and gain profound insights into global energy consumption patterns over time."

Metadata: https://www.kaggle.com/datasets/anshtanwar/global-data-on-sustainable-energy

### Data Cleaning Issues:
* **One row per country per year** (176 countries, 2000–2020; 3,649 rows). The same country appears up to 21 times, so rows are not independent. For a comparison between countries, select one year first.
* **The population density column is broken in two ways.** Its name contains the characters `\n` (`Density\n(P/Km2)`), and it loads as text because some values have commas (`2,239`). The code below renames it to `density_per_km2` and converts it to numbers.
* **Missing values differ a lot by column:** `Renewables (% equivalent primary energy)` is blank for 59% of rows and `Financial flows to developing countries (US $)` for 57%; `gdp_per_capita` is blank for 8%. Countries with missing values are not a random subset (they tend to be smaller or poorer), so check which countries drop out of your analysis.
* Density, land area, latitude and longitude are a single value per country, repeated for every year.
* Field names are overly long; `relabeled` can shorten the ones you use.

### Possible Hypotheses:
Can investigate trends over time, differences in means between countries, correlation between fields -- many possibilities! For example, one could look for differences between the energy habits of richer (gdp_per_capita) and poorer nations.

In [ ]:
filename = "data/global-data-on-sustainable-energy.csv"
energy = Table.read_table(filename)

# Fix the population density column: awkward name, and text values such as '2,239'
old_label = energy.labels[17]
print('Original column name:', old_label)

def text_to_density(text):
    """Remove thousands commas ('2,239' -> 2239.0); missing values become nan."""
    if text == 'nan':
        return np.nan
    return float(text.replace(',', ''))

energy = energy.with_column('density_per_km2', energy.apply(text_to_density, old_label)).drop(old_label)
energy.show(3)

# Example: one year, so each country appears once
energy_2019 = energy.where('Year', 2019)
energy_2019.num_rows

---
# Earthquakes in the Eastern US and Canada (DS6)
The eastern part of North America does not have nearly as many earthquakes as California, but they do happen: a magnitude 4.8 earthquake in New Jersey on 5 April 2024 was felt across Philadelphia. The data here are from the US Geological Survey earthquake catalog for the region shown on the map below, from 1925 to April 2024. The region includes part of eastern Canada (259 of the earthquakes).
More information: https://earthquake.usgs.gov/earthquakes/map

<img src="data/earthquake_extraction_region.jpeg">

### Data Cleaning Issues:

* **Not every event is an earthquake.** The `type` column shows that 213 of the 1,506 events are explosions (189, mostly mine blasts in West Virginia), rock bursts, or mine collapses. The code below keeps only earthquakes; the full table stays available as `events`.
* **The early record is incomplete.** Only 36 events are from before 1970, because there were few seismometers, not few earthquakes. For counting or comparing, use 1980 onward (1,165 earthquakes), as in the code below.
* **Small earthquakes are left out.** The catalog was downloaded with a minimum magnitude of 2.5, so almost nothing smaller appears. This matters: the *mean* magnitude of a region mostly reflects that cutoff, so comparing mean magnitudes between states tells you very little. Compare *counts* or *rates* instead.
* **Magnitude is measured several ways.** The `magType` column lists about a dozen magnitude scales (`md`, `mblg`, `ml`, `mw`, ...). They roughly agree for earthquakes of this size, but they are not identical; mention this in your limitations.
* **Location text.** `place` looks like `2 km NE of Ossipee, New Hampshire`. The code below takes the text after the last comma as the region. For 48 events there is no comma (e.g., `Kentucky-Tennessee border region`), so the whole text is used. Check the region names before relying on them: a few named events are labeled differently (the April 2024 earthquake is `2024 Whitehouse Station, New Jersey Earthquake`, which gives the region `New Jersey Earthquake`).
* There are missing values in many of the other columns (depth, error estimates).

### Possible Hypotheses:
* **The Gutenberg–Richter law.** Seismologists find that the number of earthquakes at or above magnitude *M* falls by about a factor of 10 for each unit of magnitude: log₁₀(N) = a − b·M, with b close to 1. Does that hold here? The code below counts earthquakes at or above each magnitude. Fit a regression line to log₁₀(count) versus magnitude, estimate b, and put a bootstrap confidence interval on it. Is b different for different regions or time periods?
* **Rates by region.** Tennessee has more than twice as many cataloged earthquakes since 1980 as South Carolina. Is the difference in earthquakes per decade larger than chance variation would explain?
* **Change over time.** Has the number of earthquakes per year (magnitude 2.5 or above) changed since 1980? You will need to argue that detection of earthquakes this size was steady over the whole period.
* **Where are the non-earthquakes?** Use `events` to look at where and when explosions and mine collapses occur.

In [ ]:
url = 'data/east_coast_earthquakes.csv'
events = Table.read_table(url)      # everything, including explosions and mine collapses
events.group('type').show()

# Keep only earthquakes
eq = events.where('type', 'earthquake')

# Year from the time text, e.g. '1925-10-09T13:55:00.000Z' -> 1925
def year_of(time_text):
    return int(time_text[:4])

# Region from the place text, e.g. '2 km NE of Ossipee, New Hampshire' -> 'New Hampshire'
def region_of(place_text):
    return place_text.split(', ')[-1]

eq = eq.with_columns('year', eq.apply(year_of, 'time'),
                     'region', eq.apply(region_of, 'place'))

# Use 1980 onward, when the record is reasonably complete above magnitude 2.5
eq = eq.where('year', are.above_or_equal_to(1980))
print('Earthquakes since 1980:', eq.num_rows)

# How many earthquakes are at or above each magnitude? (for the Gutenberg-Richter question)
magnitudes = np.arange(2.5, 5.1, 0.5)
counts = make_array()
for m in magnitudes:
    counts = np.append(counts, np.count_nonzero(eq.column('mag') >= m))

gr = Table().with_columns('magnitude', magnitudes, 'count at or above', counts)
gr = gr.with_column('log10 count', np.log10(gr.column('count at or above')))
gr.show()

---
# Stream Gauge Data for the Pennypack Creek in Philadelphia (DS7)
The US Geological Survey runs gauges on thousands of US streams. This one is on Pennypack Creek at Pine Road in Philadelphia (USGS site 01467042), the same creek you studied in the Pennypack mini project. The file has a reading **every 15 minutes for a full water year**, 1 October 2023 to 30 September 2024: stream flow all year, and water temperature, specific conductance, dissolved oxygen, pH and turbidity whenever those sensors were in the water.

Gauge page: https://waterdata.usgs.gov/monitoring-location/01467042/ &nbsp;|&nbsp; How the file was built: `data/pennypack_wy2024_README.md`

| Column | Measurement | Units |
|---|---|---|
| `datetime` | time of the reading (Eastern Standard Time all year) | YYYY-MM-DD HH:MM |
| `discharge_cfs` | discharge (stream flow) | cubic feet per second |
| `gage_height_ft` | gage height (water level) | feet |
| `temperature_C` | water temperature | °C |
| `specific_conductance_uS_cm` | specific conductance, a measure of dissolved salts | microsiemens per cm |
| `dissolved_oxygen_mg_L` | dissolved oxygen | mg/L |
| `dissolved_oxygen_pct_sat` | dissolved oxygen | percent of saturation |
| `pH` | pH | standard units |
| `turbidity_FNU` | turbidity (cloudiness from suspended particles) | formazin nephelometric units |
| `turbidity_below_limit` | True when the true value is below the sensor's lower limit of 0.3 | True/False |

### Data Cleaning Issues:
* **Readings are not independent.** This is the most important thing to understand about this dataset. Each reading is nearly identical to the one 15 minutes before it (the correlation between consecutive readings is 0.94 or higher for every measurement). There are 35,136 rows, but nowhere near 35,136 independent observations. A permutation test or bootstrap that treats every row as independent will give a tiny p-value or a very narrow confidence interval for almost anything. Use **days** or **storms** as your units instead: the code below builds a one-row-per-day table.
* **Storms dominate.** Median discharge is 43 cfs, but the peak is 4,930 cfs, more than 100 times larger. Daily peak flow exceeded 200 cfs on 50 days. Use logarithms for discharge and turbidity (the code adds `log10_discharge`), and remember that averages are pulled around by a handful of storms.
* **The water-quality sensors were removed for the winter.** USGS pulls them out because they ice over. Temperature, conductance, oxygen, pH and turbidity are blank from 4 December 2023 to 29 February 2024; discharge is recorded all year. This matters for one question in particular: road salt raises specific conductance in winter, and the sensor is out of the water for exactly those months. The data are missing *when the interesting thing happens*, not at random.
* **Discharge is calculated from gage height.** USGS measures the water level and converts it to flow with a calibration curve. Their close relationship is built in; it is not a discovery.
* **Turbidity has a floor.** 142 readings are at the sensor's lower limit (0.3 FNU); `turbidity_below_limit` marks them. The true values are lower.
* Dissolved oxygen percent saturation is also blank from 1 October to 10 November 2023.
* All times are Eastern Standard Time, including in summer, so there is no repeated or skipped hour when the clocks change. In summer, clock time is one hour later than the time shown. For the same reason the file starts at 23:00 on 30 September 2023, so that date has only four readings.

### Possible Hypotheses:
* **Turbidity and discharge.** How does turbidity change with flow? Fit a regression on log scales. Then look inside a single storm: does turbidity peak before the flow does, at the same time, or after? Is that pattern the same in every storm?
* **Dilution.** Specific conductance drops when storm water arrives (the lowest value of the year, 91 µS/cm, came at a flow of 3,890 cfs). How well does `log10_discharge` predict conductance? Is the relationship the same in fall as in spring and summer?
* **The daily oxygen cycle.** Dissolved oxygen is lowest around 6 a.m. and highest around 4 p.m. Why? Is the daily swing (each day's maximum minus minimum) larger in July than in November? Here a *day* is a natural unit of observation.
* **Temperature and oxygen.** Cold water holds more oxygen. How strong is the relationship between daily mean temperature and daily mean dissolved oxygen? Which days fall far from the line, and are they storm days?
* **How fast does the creek rise and fall?** Compare how long the stream spends rising with how long it spends falling. What does that say about a stream in a city, where rain runs off pavement?
* **What is your sample size, really?** Pick one relationship and estimate it three ways: using every 15-minute reading, using daily values, and using one value per storm. Compare the confidence intervals. Which one is honest, and why?

**Link to the mini project:** specific conductance responds to dissolved salts, including the chloride you analyzed in the mini project. Because the sensor is removed in winter to protect it from ice, this file cannot show winter road-salt pulses. If your group wants to study salt, say what data you would need and why this gauge cannot provide it.

In [ ]:
pp = Table.read_table('data/pennypack_wy2024.csv')
pp.show(3)

# Date, month and hour from the time text, e.g. '2023-12-18 08:15'
def date_of(time_text):
    return time_text[:10]

def month_of(time_text):
    return time_text[:7]

def hour_of(time_text):
    return int(time_text[11:13])

pp = pp.with_columns('date', pp.apply(date_of, 'datetime'),
                     'month', pp.apply(month_of, 'datetime'),
                     'hour', pp.apply(hour_of, 'datetime'))

# Discharge spans a huge range, so work with its logarithm
pp = pp.with_column('log10_discharge', np.log10(pp.column('discharge_cfs')))

# One row per day: peak and mean discharge (nanmax and nanmean skip the few missing readings)
flow = pp.select('date', 'discharge_cfs')
daily = flow.group('date', np.nanmax).relabeled('discharge_cfs nanmax', 'peak_cfs')
daily = daily.with_column('mean_cfs', flow.group('date', np.nanmean).column('discharge_cfs nanmean'))

# Call a day a "storm day" if its peak flow is above 200 cfs (a choice; try other cutoffs)
daily = daily.with_column('storm_day', daily.column('peak_cfs') > 200)
print('Days:', daily.num_rows, '  Storm days:', np.count_nonzero(daily.column('storm_day')))
daily.sort('peak_cfs', descending=True).show(5)

# Only the readings taken while the water-quality sensors were in the water
# (a comparison with a blank value is False, so this drops the winter gap)
wq = pp.where('specific_conductance_uS_cm', are.above(0))
print('Readings with water-quality data:', wq.num_rows, 'of', pp.num_rows)

---
# Philadelphia Daily Weather, 1940–2025 (DS8)
Is Philadelphia getting warmer? Wetter? Less snowy? This dataset has the official daily weather record from **Philadelphia International Airport** from 1 July 1940 to 31 December 2025: 86 years, one row per day. It comes from NOAA's Global Historical Climatology Network, the same kind of record climate scientists use to measure long-term change.

How the file was built, and known issues: `data/philly_weather_daily_README.md`

| Column | Meaning | Units |
|---|---|---|
| `date` | calendar date | YYYY-MM-DD |
| `tmax_C`, `tmin_C` | highest and lowest temperature of the day | °C |
| `precip_mm` | precipitation (rain plus melted snow) | mm |
| `snowfall_mm` | new snow that fell | mm |
| `snow_depth_mm` | snow on the ground | mm |
| `wind_m_s` | average wind speed (1984 onward) | meters per second |
| `precip_trace`, `snow_trace` | True when some fell but too little to measure (recorded as 0) | True/False |

To convert: °F = °C × 9/5 + 32; inches = mm ÷ 25.4.

### Data Cleaning Issues:
* **Two years are incomplete.** The record starts in July 1940, and there are no observations from 23 June to 30 November 1945. Yearly totals and averages for 1940 and 1945 cannot be compared with other years. The code below keeps 1946–2025, which gives 80 complete years.
* **Days are not independent.** Today's high temperature is closely related to yesterday's (correlation 0.92), and every year has the same seasonal cycle. For questions about change over time, summarize each year into one number first (as the code below does); years are much closer to independent than days.
* **One station is not the whole region.** The airport and the city around it grew over these 86 years, and instruments were replaced. Some of any warming at this site may come from the urban heat island or equipment changes rather than regional climate. Say this in your limitations.
* **Wind speed starts in 1984**; it is blank before then.
* **Trace amounts.** On 4,375 days precipitation fell but was too little to measure. It is recorded as 0, with `precip_trace` True. Decide whether those count as rainy days.
* **A winter spans two calendar years.** For snowfall, add up July through June, not January through December.

### Possible Hypotheses:
* **Is it getting warmer?** Fit a regression line to yearly mean temperature against year and put a bootstrap confidence interval on the slope. Or compare the first ten years with the last ten. How many degrees per decade?
* **Are winters or summers changing faster?** Compare the trend in January low temperatures with the trend in July high temperatures.
* **Hot days and freezing days.** Has the number of days per year reaching 32 °C (about 90 °F) changed? The number of days falling to freezing?
* **Is heavy rain more frequent?** Count the days each year with 25 mm (about an inch) or more of precipitation. Yearly counts vary a lot; is there a trend you can distinguish from chance?
* **Is it snowing less?** Total the snowfall for each winter. Snowfall varies enormously from one winter to the next (from none at all in 1972–73 to 2 meters in 2009–10). What does that much variation do to your ability to detect a trend?
* **Your own year.** How unusual was the year you were born, or last year, compared with the whole record?

In [ ]:
weather = Table.read_table('data/philly_weather_daily.csv')
weather.show(3)

# Year and month from the date text, e.g. '1996-01-07'
def year_of(date_text):
    return int(date_text[:4])

def month_of(date_text):
    return int(date_text[5:7])

weather = weather.with_columns('year', weather.apply(year_of, 'date'),
                               'month', weather.apply(month_of, 'date'))

# Daily mean temperature: halfway between the day's high and low
weather = weather.with_column('tmean_C', (weather.column('tmax_C') + weather.column('tmin_C')) / 2)

# Keep the 80 complete years (1940 and 1945 are missing months of data)
complete = weather.where('year', are.above_or_equal_to(1946))

# One row per year: mean temperature and total precipitation
yearly = complete.select('year', 'tmean_C').group('year', np.mean).relabeled('tmean_C mean', 'mean_temp_C')
yearly = yearly.with_column('total_precip_mm',
                            complete.select('year', 'precip_mm').group('year', sum).column('precip_mm sum'))
yearly.show(3)
yearly.scatter('year', 'mean_temp_C')

---
# CAMELS Hydrology Dataset (DS19)

The CAMELS (Catchment Attributes and Meteorology for Large-sample Studies) dataset is an important benchmark dataset for hydrological modeling. For the final project, a curated subset of the climatic, hydrological, topographic, and vegetation characteristics of 671 watersheds across the US has been selected. The data are from a wide variety of sources, but the hydrological attributes are from the US Geological Survey. The attributes were downloaded from https://ral.ucar.edu/solutions/products/camels 

You can read more about the dataset in the peer-reviewed paper here: https://hess.copernicus.org/articles/21/5293/2017/ 

There are two files that will be useful:
- The description of columns and data sources can be found in `data/camels_attributes_eds.csv`
- The data itself can be found in `data/camels_assembled.csv`

### Data Cleaning Issues

* **One watershed has no streamflow values.** Gauge 3281100 (Kentucky) is missing `q_mean`, `runoff_ratio`, `q5`, `q95`, and `zero_q_freq`. The code below removes it.
* **12 watersheds have a runoff ratio above 1**, meaning more water leaves in the stream than falls as precipitation. Over many years that is not physically possible. Ten of the twelve are in the mountains of western Washington, with one each in Colorado and coastal Oregon: places where precipitation is hard to measure (few gauges, steep terrain, snow) and is probably underestimated. These are measurement problems, not errors in the file. Decide whether to keep them, and say what you did; the code below adds a column that flags them.
* **Many dry-region streams stop flowing:** `zero_q_freq` is above 0 for 166 watersheds, and `q5` is 0 or nearly 0 for many of them. Ratios and logarithms of these values need care.
* Watersheds near each other share climate and geology, so they are not independent observations.

### Possible Hypotheses
Such a large data set opens up many possibilities. How does aridity affect mean runoff? How does precipitation affect low flows (e.g., Q5, the 5th percentile of daily runoff, or zero_q_freq, the % of days with no streamflow)? How does snow cover affect the runoff ratio (ratio runoff/precipitation)? Do forested watersheds have higher low flows than non-forested watersheds? Do arid (aridity >= 1) watersheds have different runoff ratio than humid (aridity < 1) watersheds?

You will find lots of scatter in your data. KNN regression could be a useful tool. Linear regression may work in some cases, but first, evaluate whether the relation you are trying to predict is justifiably linear. If you are going to do a hypothesis test, try to pick a physically meaningful test statistic. 


In [ ]:
attributes = Table.read_table('data/camels_attributes_eds.csv')
attributes

In [ ]:
camels = Table.read_table('data/camels_assembled.csv')

# Remove the one watershed with no streamflow values (a comparison with nan is always False)
camels = camels.where('q_mean', are.above_or_equal_to(0))

# Flag watersheds where runoff exceeds precipitation (runoff ratio above 1)
camels = camels.with_column('runoff_exceeds_precip', camels.column('runoff_ratio') > 1)
print('Watersheds:', camels.num_rows, '  with runoff ratio above 1:', np.count_nonzero(camels.column('runoff_exceeds_precip')))
camels

---
---
## **General Datasets**  <a id="General"></a>

---
# Student Performance in Secondary Education (DS18)

## Dataset Overview

This dataset examines factors that influence student achievement in secondary education (high school) from two Portuguese schools. The data was collected using school reports and student questionnaires, and includes information about student grades, demographics, family background, and school-related factors.

**Source:** UCI Machine Learning Repository  
**Original Study:** Cortez & Silva (2008), "Using Data Mining to Predict Secondary School Student Performance"  
**Sample Size:** 395 students (Mathematics course) or 649 students (Portuguese language course)  
**Time Period:** Data collected from two Portuguese secondary schools (Gabriel Pereira and Mousinho da Silveira)

## Why This Dataset Matters

Understanding what factors contribute to student success is crucial for educators, policymakers, and families. This dataset allows us to explore questions like:

- Does having internet access at home affect academic performance?
- Do students with parental support perform differently than those without?
- How does study time relate to final grades?
- Are there differences in achievement based on gender, family structure, or socioeconomic factors?

By analyzing this data, we can identify factors that may help or hinder student success, which could inform school policies and support programs.



## Dataset Variables (33 total)

### Student Demographics
- **age** - Student's age (15-22 years)
- **sex** - Gender (M=male, F=female)
- **address** - Home location (U=urban, R=rural)
- **famsize** - Family size (LE3 = ≤3 members, GT3 = >3 members)

### Family Background
- **Medu** - Mother's education level (0=none, 1=primary, 2=5th-9th grade, 3=secondary, 4=higher education)
- **Fedu** - Father's education level (same scale)
- **Mjob** - Mother's occupation (teacher, health, services, at_home, other)
- **Fjob** - Father's occupation (same categories)
- **Pstatus** - Parent cohabitation status (T=living together, A=apart)
- **guardian** - Student's guardian (mother, father, or other)

### School-Related Factors
- **school** - Which school (GP=Gabriel Pereira, MS=Mousinho da Silveira)
- **traveltime** - Home to school travel time (1=<15 min, 2=15-30 min, 3=30min-1hr, 4=>1hr)
- **studytime** - Weekly study time (1=<2hrs, 2=2-5hrs, 3=5-10hrs, 4=>10hrs)
- **failures** - Number of past class failures (0-4)
- **schoolsup** - Extra educational support (yes/no)
- **famsup** - Family educational support (yes/no)
- **paid** - Extra paid classes/tutoring (yes/no)
- **activities** - Participates in extracurricular activities (yes/no)
- **nursery** - Attended nursery school (yes/no)
- **higher** - Wants to pursue higher education (yes/no)
- **internet** - Internet access at home (yes/no)
- **romantic** - In a romantic relationship (yes/no)

### Personal & Social Factors
- **famrel** - Quality of family relationships (1=very bad to 5=excellent)
- **freetime** - Free time after school (1=very low to 5=very high)
- **goout** - Going out with friends (1=very low to 5=very high)
- **Dalc** - Workday alcohol consumption (1=very low to 5=very high)
- **Walc** - Weekend alcohol consumption (1=very low to 5=very high)
- **health** - Current health status (1=very bad to 5=very good)
- **absences** - Number of school absences (0-93 days)

### **Academic Outcomes** (Our Target Variables!)
- **G1** - First period grade (0-20 scale)
- **G2** - Second period grade (0-20 scale)
- **G3** - **Final grade (0-20 scale)** ← **PRIMARY OUTCOME**


## Research Questions to Explore

### Comparison Questions
- Do male and female students achieve different mean final grades?
- Does having internet access at home correlate with better academic performance?
- Do students receiving extra educational support perform differently than those who don't?
- Is there a difference in achievement between urban and rural students?

### Correlation Questions
- Is there a relationship between weekly study time and final grades?
- Does parental education level correlate with student achievement?
- How do school absences relate to academic performance?

### Predictive Questions
- Can we predict final grades (G3) from first period grades (G1)?
- What combination of factors best predicts student success?
- Can we identify students at risk of failing based on early indicators?

---

## Important Considerations

### Statistical Notes
- **Correlation ≠ Causation:** If two variables are correlated, it doesn't mean one causes the other. For example, students with internet access may perform better, but this could be because wealthier families have both internet and other educational advantages.

- **Multiple Testing:** When testing many variables, some relationships may appear significant by chance. Be careful about drawing strong conclusions from exploring 30+ variables.

- **Effect Sizes:** A statistically significant result isn't always practically meaningful. Always consider the magnitude of differences, not just p-values.

### Ethical Considerations
- This data represents real students and their educational experiences
- Avoid making stereotypical generalizations about groups
- Consider systemic and structural factors, not just individual characteristics
- Think about how findings could be used to help students, not label or limit them

---

## How to Load the Data

```python
from datascience import *
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# Load the Mathematics dataset
# Important: This CSV uses semicolons (;) as separators, not commas!
students = Table.read_table('student-mat.csv', sep=';')

# Display basic information
print(f"Number of students: {students.num_rows}")
print(f"Number of variables: {students.num_columns}")

# Show first few rows
students.show(5)
```

## Before You Analyze: Three Warnings

* **G1 and G2 nearly determine G3.** The three grades are the same students in the same course, a few months apart: the correlation between G2 and G3 is 0.90 (0.97 among students with a nonzero final grade). A model that "predicts" G3 from G1 or G2 will look excellent and tell you nothing about *why* students succeed. If your question is about study time, family support, or other factors, leave G1 and G2 out of your predictors. Using them is only appropriate if your question is specifically about early grades as a warning sign.
* **A final grade of 0 does not mean a score of zero.** 38 of the 395 students have G3 = 0, and every one of them has 0 recorded absences, although all of them earned a first-period grade (between 4 and 12 out of 20). These are almost certainly students who left the course or did not take the final assessment. They pull group means down sharply. Decide whether your question is about all enrolled students or about students who completed the course, analyze accordingly, and say which you chose. Reporting both is a good sensitivity check.
* **Observational survey data.** Students were not assigned to study more or to have internet at home. Differences between groups may reflect family background or school rather than the factor itself, and the data come from two schools in Portugal in 2005–2006.

## Citation

When using this dataset in your analysis, cite:

**Cortez, P., & Silva, A. M. G. (2008).** Using data mining to predict secondary school student performance. *Proceedings of 5th Annual Future Business Technology Conference*, Porto, Portugal, pp. 5-12.

**Dataset:** Cortez, P. (2008). Student Performance [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5TG7T

**License:** Creative Commons Attribution 4.0 International (CC BY 4.0) - Free to use with attribution



## Let's Begin!

Now that you understand the dataset, you're ready to explore the data and test hypotheses about factors that influence student success!

In [9]:
# Load the Mathematics dataset
# Important: This CSV uses semicolons (;) as separators, not commas!
students = Table.read_table('data/student-mat.csv', sep=';')

# Display basic information
print(f"Number of students: {students.num_rows}")
print(f"Number of variables: {students.num_columns}")

# Show first few rows
students.show(5)


Number of students: 395
Number of variables: 33


school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,reason,guardian,traveltime,studytime,failures,schoolsup,famsup,paid,activities,nursery,higher,internet,romantic,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
GP,F,18,U,GT3,A,4,4,at_home,teacher,course,mother,2,2,0,yes,no,no,no,yes,yes,no,no,4,3,4,1,1,3,6,5,6,6
GP,F,17,U,GT3,T,1,1,at_home,other,course,father,1,2,0,no,yes,no,no,no,yes,yes,no,5,3,3,1,1,3,4,5,5,6
GP,F,15,U,LE3,T,1,1,at_home,other,other,mother,1,2,3,yes,no,yes,no,yes,yes,yes,no,4,3,2,2,3,3,10,7,8,10
GP,F,15,U,GT3,T,4,2,health,services,home,mother,1,3,0,no,yes,yes,yes,yes,yes,yes,yes,3,2,2,1,1,5,2,15,14,15
GP,F,16,U,GT3,T,3,3,other,other,home,father,1,2,0,no,yes,yes,no,yes,yes,no,no,4,3,2,1,2,5,4,6,10,10


---
# Philadelphia Open Data School Graduation Rates (DS11)
This longitudinal open data file includes information about the graduation rates for schools broken out by: graduation rate type (four-year, five-year, or six-year), demographic category (EL status, IEP status, Economically Disadvantaged Status, Gender, or Ethnicity), and ninth-grade cohort. Students are attributed to the last school at which they actively attended in the respective graduation window, which ends on September 30 each year. Students are classified as EL, as having an IEP, and/or economically disadvantaged if they were designated as such at any point during their high school career.
see: https://www.philasd.org/performance/programsservices/open-data/school-performance/#school_graduation_rates 
see also: https://www.philasd.org/research/wp-content/uploads/sites/90/2020/05/graduation-rate-definitions-and-trends-may-2020.pdf

The 'group' column can be used with the `.where()` Table method to select a particular comparison such as Economically Disadvantaged which then has two subgroups for comparison.

### Data Cleaning Issue:
Some of the fields have mixed numerical and text data (e.g., num, score), with the code "s" where a score was not calculated.
<br>A dataset with the score computed as a floating point number but with the Table rows with "s" removed is available for analysis:<br>
```python
fscore_grad = 'data/Philly_grad_fscore.csv'
```
### What is missing, and why it matters
* **Suppressed scores are not missing at random.** A score is replaced by "s" to protect student privacy when a group is very small (fewer than 10 students) or when showing it would let someone work out a small group's result from the totals. In the full file, 41% of the rows (9,476 of 23,021) are suppressed.
* **Suppression depends on the group.** About 13% of "All Students" rows are suppressed, compared with 66% of English-learner rows, 70% of Asian-student rows, 95% of Multi Racial/Other rows, and 100% of American Indian/Alaskan Native and Native Hawaiian/Pacific Islander rows. The scores you can see for smaller groups come only from the schools where that group is large, which are not typical schools for that group.
* So `Philly_grad_fscore.csv` (suppressed rows removed) is not a random sample of schools and groups. An average over the remaining rows describes large groups at large schools; say this in your limitations, and avoid comparisons where most of one group's rows are suppressed.
* **Rows are not independent.** The same students are counted in the 4-, 5-, and 6-year rates and in several groups (for example Gender and Ethnicity), and the same school appears in every cohort. Select one `rate_type` and one `group` before comparing.
* An average of school rates is not the district's rate: it counts a school of 30 students the same as a school of 600. Weight by `denom` if you want a rate for students rather than for schools.

### Possible Hypotheses
Questions this file can support (use `Philly_grad_fscore.csv`, and select one `rate_type` and one `group` first):

* **Change over time.** For "All Students", has the 4-year graduation rate of schools changed between the 2010–2011 and 2017–2018 cohorts? The same schools appear in each cohort, so think about comparing each school with itself rather than treating the cohorts as separate samples.
* **Does extra time matter?** For the same school and cohort, how much higher is the 6-year rate than the 4-year rate? Give an estimate with a confidence interval. (6-year rates exist only through the 2015–2016 cohort.)
* **Gender.** Is the 4-year rate different for female and male students? Both subgroups are reported for most schools, so compare within schools.
* **Economic disadvantage.** Is the 4-year rate different for economically disadvantaged students? About a quarter of these rows are suppressed; check which schools drop out before interpreting the result.
* **District vs. alternative schools.** The `sector` column separates regular District schools from Alternative schools, whose rates are far lower. Before interpreting that gap, find out who alternative schools serve: students are not assigned to them at random.

Questions to avoid: comparisons among ethnicity subgroups or between English learners and other students. Most of those rows are suppressed (56% to 100%, see above), so the scores you can see are not representative of those groups.

Keep in mind that this is a limited data set covering a socially sensitive topic. These are rates for schools, not measurements of individual students, and they cannot show *why* rates differ. Do not draw broad conclusions about groups of students.

In [ ]:
url = "https://cdn.philasd.org/offices/performance/Open_Data/School_Performance/Graduation_Rates/SDP_Graduation_Rates_School_S_2022-05-23.csv"
grad = Table.read_table(url)
grad.show(3)

In [ ]:
fscore_grad = 'data/Philly_grad_fscore.csv'
fgrad = Table.read_table(fscore_grad)
fgrad.show(3)

---
# Crime Data for Philadelphia (DS13)
The data came from OpenDataPhilly: https://opendataphilly.org/datasets/crime-incidents
Each row is one reported incident dispatched by the Philadelphia Police Department in 2023 (169,017 rows).

### Data Cleaning Issues:
* **Two kinds of time.** `dispatch_date` and `dispatch_time` are Philadelphia local time. `dispatch_date_time` is in UTC (it ends in `+00`), which is 4 or 5 hours ahead of local time. Use the local columns.
* **The year boundary is off by a few hours.** The file was cut using the UTC timestamp, so it starts with 56 incidents from the evening of 31 December 2022 (local time) and is missing incidents after about 7 p.m. on 31 December 2023. The code below keeps only local 2023 dates; remember that 31 December 2023 is incomplete.
* **Do not use the `hour` column.** It is blank for 100,619 rows (60%), and where present it is sometimes an hour off from `dispatch_time`. The code below makes a `local_hour` column from `dispatch_time` instead.
* These are *reported* incidents. Crimes that are never reported are missing, and reporting rates differ by type of crime.
* Months have different numbers of days; compare months using incidents **per day**, not raw monthly counts.
* To extract months you need to parse the date text (e.g., `2023-03-11`). Working with time series data can be tricky using tables; look at Lab 04 for useful functions.

### Possible Hypotheses:
Possible correlation: type of crime and time of day. Could look at whether a particular type of crime occurs more frequently at a particular time of day, or whether the number of crimes per day is significantly different in different months.

In [ ]:
url = 'data/Philly_crime_2023.csv'
crime = Table.read_table(url)

# Keep only incidents dispatched in 2023, local time (drops 56 from the evening of 2022-12-31)
crime = crime.where('dispatch_date', are.containing('2023-'))

# Hour of day (0-23) from the local dispatch time, e.g. '17:12:00' -> 17
def hour_of(time_text):
    return int(time_text[:2])

crime = crime.with_column('local_hour', crime.apply(hour_of, 'dispatch_time'))
crime.show(3)

---
# Motor Vehicle Crashes on Staten Island, 2012–2024 (DS15)

This data set comes from New York City's **Motor Vehicle Collisions – Crashes** table (NYC Open Data, also listed on Data.gov). The citywide file is huge, so it has been trimmed to crashes whose `BOROUGH` is Staten Island. Despite its file name, it covers **1 July 2012 to 6 April 2024**: 60,109 crashes, of which 2,724 are from 2023.

https://catalog.data.gov/dataset/motor-vehicle-collisions-crashes

"The Motor Vehicle Collisions crash table contains details on the crash event. Each row represents a crash event. The Motor Vehicle Collisions data tables contain information from all police reported motor vehicle collisions in NYC. The police report (MV104-AN) is required to be filled out for collisions where someone is injured or killed, or where there is at least $1000 worth of damage (https://www.nhtsa.gov/sites/nhtsa.dot.gov/files/documents/ny_overlay_mv-104an_rev05_2004.pdf). It should be noted that the data is preliminary and subject to change when the MV-104AN forms are amended based on revised crash details. For the most accurate, up to date statistics on traffic fatalities, please refer to the NYPD Motor Vehicle Collisions page (updated weekly) or Vision Zero View (updated monthly)."

### Data Cleaning Issues
* `Unnamed: 0` is a leftover row index from when the file was trimmed; the code below drops it.
* Dates are text in `MM/DD/YYYY` form. The code below shows how to keep a single year.
* Some fields have missing values, especially the street names and the contributing factors for vehicles 3–5.
* **Counts over time are not directly comparable.** Crashes per year in this file:

| Years | Crashes per year |
|---|---|
| 2013–2018 | about 6,200–8,300 |
| 2019 | 3,650 |
| 2020–2023 | about 2,400–2,700 |

  Before using this as a trend, find out why. Two things are known to be going on:
  1. **Missing borough labels.** Many crash records citywide have a blank `BOROUGH` even though their location is on Staten Island, and those crashes are *not* in this file. From 2016 on, roughly 40–50% of the crashes located on Staten Island have no borough label (about 11% in 2013). So this file is not every Staten Island crash, and the share it leaves out changes from year to year.
  2. **A citywide drop in 2020.** Across all of NYC, reported crashes fell from about 211,000 in 2019 to about 113,000 in 2020 and stayed near that level. The COVID-19 pandemic is the obvious candidate, but it does not explain why counts stayed low afterward, or why Staten Island's count fell in 2019.
* Only police-reported crashes are included (injury, death, or at least $1,000 damage), so minor crashes are missing.
* Deaths are rare (fewer than 15 per year in this file), so comparisons of fatality rates will be based on small counts.

### Possible Hypotheses
Such a large data set opens up many possibilities. Compare percent of crashes resulting in injuries by vehicle type? How about in two-vehicle crashes? Are certain months or times of day statistically more likely to have crashes? Certain zip codes (you may need population data to convert to per capita)? If you study a change over time, explain how you dealt with the borough-labeling issue above.


In [ ]:
filename = 'data/StatenIsland_crash_data_2023.csv'
crash = Table.read_table(filename)

# Drop the leftover index column
crash = crash.drop('Unnamed: 0')
crash.show(3)

# Example: keep only crashes from one year (dates look like 12/13/2021)
crash_2023 = crash.where('CRASH DATE', are.containing('/2023'))
crash_2023.num_rows